In [1]:
# (1) Reading Comments from YouTube Key
import os
import requests
import pandas as pd


API_KEY = os.getenv('YOUTUBE_API_KEY')


def get_video_info(video_id):
    url = 'https://www.googleapis.com/youtube/v3/videos'

    params = {
        'key': API_KEY,
        'id': video_id,
        'part': 'snippet,statistics'
    }

    response = requests.get(url, params = params)
    data = response.json()

    if not data.get('items'):
        print('Video not found!')
        return None

    item = data['items'][0]

    video_info = {
        'video_id': video_id,
        'title': item['snippet']['title'],
        'channel': item['snippet']['channelTitle'],
        'views': item['statistics'].get('viewCount', 0),
        'likes': item['statistics'].get('likeCount', 0)
    }

    print(f"Video: {video_info['title']}")
    print(f"Channel: {video_info['channel']}")
    print(f"Views: {video_info['views']} | Likes: {video_info['likes']}")

    return video_info


def get_comments(video_id, max_comments = 200):
    url = 'https://www.googleapis.com/youtube/v3/commentThreads'

    comments = []
    next_page = None

    print('\nFetching comments...')

    while len(comments) < max_comments:
        params = {
            'key': API_KEY,
            'videoId': video_id,
            'part': 'snippet',
            'maxResults': 100,
            'textFormat': 'plainText'
        }

        if next_page:
            params['pageToken'] = next_page

        response = requests.get(url, params = params)
        data = response.json()

        if 'error' in data:
            print(f"Error: {data['error']['message']}")
            break

        for item in data.get('items', []):
            comment = item['snippet']['topLevelComment']['snippet']

            comments.append({
                'comment_id': item['id'],
                'video_id': video_id,
                'author': comment.get('authorDisplayName', ''),
                'text': comment.get('textDisplay', ''),
                'likes': comment.get('likeCount', 0),
                'date': comment.get('publishedAt', '')
            })

            if len(comments) >= max_comments:
                break

        next_page = data.get('nextPageToken')

        if not next_page:
            break

    comments_df = pd.DataFrame(comments)

    print(f'Fetched {len(comments_df)} comments ✓')

    return comments_df


def save_comments(comments_df, video_id):
    os.makedirs('data', exist_ok = True)

    file_path = f'data/{video_id}_comments.csv'

    comments_df.to_csv(file_path, index = False, encoding = 'utf-8-sig')

    print(f'Saved to: {file_path}')

    return file_path


if __name__ == '__main__':
    VIDEO_ID = 'Aj-pyJF6ckU'

    video_info = get_video_info(VIDEO_ID)
    comments_df = get_comments(VIDEO_ID, max_comments = 200)

    save_comments(comments_df, VIDEO_ID)

    print('\nFirst 3 comments:')
    print(comments_df[['author', 'text']].head(3).to_string(index = False))

Video: Arab Idol - الأداء - محمد عساف - على الكوفية
Channel: Arab Idol
Views: 129840950 | Likes: 659092

Fetching comments...
Fetched 200 comments ✓
Saved to: data/Aj-pyJF6ckU_comments.csv

First 3 comments:
       author                                                       text
 @theyjustart                                                       2025
    @48younis محمد عساف, هو كنز, وثروه وطنيه وقوميه عربيه, والله الموفق.
@amalwesammur                                                Hhii I mm n


In [2]:
# (2) Cleaning YouTube Comments
import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_spam(text):
    spam_words = [
        'subscribe',
        'sub4sub',
        'follow me',
        'check my channel',
        'free gift',
        'earn money',
        'click here',
        'buy now']

    text_lower = str(text).lower()
    return any(word in text_lower for word in spam_words)

def clean_dataframe(df):
    df = df.copy()
    df['clean_text'] = df['text'].apply(clean_text)
    df['is_spam'] = df['text'].apply(is_spam)
    before = len(df)
    df = df[df['clean_text'].str.strip() != ''].reset_index(drop = True)
    print(f'Removed {before - len(df)} empty comments')
    print(f'Detected spam comments: {df["is_spam"].sum()}')
    print(f'Clean comments: {len(df)} ✓')
    return df

def save_cleaned(df, video_id):
    file_path = f'data/{video_id}_cleaned.csv'
    df.to_csv(file_path, index = False, encoding = 'utf-8-sig')
    print(f'Saved to: {file_path}')
    return file_path

if __name__ == '__main__':
    VIDEO_ID = 'Aj-pyJF6ckU'
    df = pd.read_csv(f'data/{VIDEO_ID}_comments.csv')
    print(f'Fetched comments: {len(df)}')
    cleaned_df = clean_dataframe(df)
    save_cleaned(cleaned_df, VIDEO_ID)
    print('\nCleaning example:')
    for _, row in cleaned_df.head(3).iterrows():
        print(f'  Before: {row["text"][:60]}')
        print(f'  After: {row["clean_text"][:60]}')
        print()

Fetched comments: 200
Removed 110 empty comments
Detected spam comments: 0
Clean comments: 90 ✓
Saved to: data/Aj-pyJF6ckU_cleaned.csv

Cleaning example:
  Before: 2025
  After: 2025

  Before: Hhii I mm n
  After: hhii i mm n

  Before: He will sing at the ceremony opening in Brasil 2014 :D 
  After: he will sing at the ceremony opening in brasil 2014 d



In [3]:
# (3) Analyzing Comment Sentiment

from transformers import pipeline
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

AR_MODEL = 'UBC-NLP/MARBERTv2'

def load_model():
    print('Loading MARBERT sentiment model...')

    tokenizer = AutoTokenizer.from_pretrained(AR_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(AR_MODEL)

    model.eval()

    sentiment_model = pipeline(
    'text-classification',
    model = model,
    tokenizer = tokenizer,
    truncation = True,
    max_length = 128,
    device = 0 if torch.cuda.is_available() else -1,
    batch_size = 32
)

    print('MARBERT model loaded successfully ✓')

    return sentiment_model

def analyze_text(model, text):
    if not text or not str(text).strip():
        return 'neutral', 0.0

    result = model(str(text))[0]

    label_map = {
        'LABEL_0': 'negative',
        'LABEL_1': 'neutral',
        'LABEL_2': 'positive'
    }

    label = label_map.get(result['label'], result['label'].lower())
    score = round(result['score'], 3)

    return label, score

def analyze_dataframe(model, df):
    labels = []
    scores = []
    total = len(df)

    for i, text in enumerate(df['clean_text']):
        label, score = analyze_text(model, text)

        labels.append(label)
        scores.append(score)

        if (i + 1) % 50 == 0:
            print(f'  {i + 1}/{total} comments...')

    df['sentiment'] = labels
    df['score'] = scores

    print(f'Analyzed {total} comments ✓')

    return df


def print_summary(df):
    total = len(df)

    positive = (df['sentiment'] == 'positive').sum()
    negative = (df['sentiment'] == 'negative').sum()
    neutral = (df['sentiment'] == 'neutral').sum()

    summary = {
        'positive': positive,
        'negative': negative,
        'neutral': neutral,
        'positive_pct': 100 * positive // total,
        'negative_pct': 100 * negative // total,
        'neutral_pct': 100 * neutral // total
    }

    print('\nSentiment Summary')
    print(f'Positive: {summary["positive"]} ({summary["positive_pct"]}%)')
    print(f'Negative: {summary["negative"]} ({summary["negative_pct"]}%)')
    print(f'Neutral : {summary["neutral"]} ({summary["neutral_pct"]}%)')

    return summary


def save_results(df, video_id):
    file_path = f'data/{video_id}_results.csv'
    df.to_csv(file_path, index = False, encoding = 'utf-8-sig')
    print(f'Saved to: {file_path}')
    return file_path

if __name__ == '__main__':
    VIDEO_ID = 'Aj-pyJF6ckU'
    df = pd.read_csv(f'data/{VIDEO_ID}_cleaned.csv')
    model = load_model()
    df = analyze_dataframe(model, df)
    print_summary(df)
    save_results(df, VIDEO_ID)
    print('\nResults examples:')
    print(df[['text', 'sentiment', 'score']].head(5).to_string(index = False))

Loading MARBERT sentiment model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were ne

MARBERT model loaded successfully ✓
  50/90 comments...
Analyzed 90 comments ✓

Sentiment Summary
Positive: 0 (0%)
Negative: 50 (55%)
Neutral : 40 (44%)
Saved to: data/Aj-pyJF6ckU_results.csv

Results examples:
                                                   text sentiment  score
                                                   2025  negative  0.545
                                            Hhii I mm n  negative  0.564
He will sing at the ceremony opening in Brasil 2014 :D    neutral  0.527
                                      FREE FALESTINE <3   neutral  0.500
He and Shakira are singing in the World Cup this year!!   neutral  0.530


In [4]:
# (4) Extracting Important Words and Recommendations
from collections import Counter

STOP_WORDS = {
    'the', 'a', 'an', 'is', 'it', 'this', 'that', 'to', 'and',
    'of', 'in', 'for', 'on', 'with', 'be', 'was', 'are', 'i',
    'you', 'he', 'she', 'we', 'they', 'not', 'have', 'had',
    'but', 'or', 'so', 'at', 'by', 'from', 'as', 'do', 'did'}

def get_top_words(df, sentiment = None, top_n = 10):
    if sentiment:
        texts = df[df['sentiment'] == sentiment]['clean_text'].dropna()
    else:
        texts = df['clean_text'].dropna()
    words = []

    for text in texts:
        for word in str(text).split():
            if word not in STOP_WORDS and len(word) > 2:
                words.append(word)

    return Counter(words).most_common(top_n)

def get_recommendations(positive_pct, negative_pct, neutral_pct):
    recommendations = []
    if positive_pct >= 60:
        recommendations.append('The content is excellent. The audience is highly satisfied, so keep the same style.')
    elif positive_pct >= 40:
        recommendations.append('The content is good, but some parts can be improved to increase audience satisfaction.')
    else:
        recommendations.append('The positive sentiment is low. Review the content style and presentation.')
    if negative_pct >= 30:
        recommendations.append('The negative sentiment is high. Check the most frequent negative words carefully.')
    if neutral_pct >= 50:
        recommendations.append('Many comments are neutral. Encourage the audience to interact more.')
    return recommendations

def print_insights(df):
    total = len(df)
    positive_pct = 100 * (df['sentiment'] == 'positive').sum() // total
    negative_pct = 100 * (df['sentiment'] == 'negative').sum() // total
    neutral_pct = 100 * (df['sentiment'] == 'neutral').sum() // total
    print('\nTop Positive Words')
    print('-----------------------------')
    for word, count in get_top_words(df, 'positive'):
        print(f'{word}: {count}')
    print('\nTop Negative Words')
    print('-----------------------------')

    for word, count in get_top_words(df, 'negative'):
        print(f'{word}: {count}')
    print('\nRecommendations')
    print('-----------------------------')
    recommendations = get_recommendations(positive_pct, negative_pct, neutral_pct)
    for recommendation in recommendations:
        print(f'- {recommendation}')
    return get_top_words(df), recommendations

if __name__ == '__main__':
    VIDEO_ID = 'Aj-pyJF6ckU'
    df = pd.read_csv(f'data/{VIDEO_ID}_results.csv')
    print_insights(df)


Top Positive Words
-----------------------------

Top Negative Words
-----------------------------
assaf: 9
bravo: 5
mohammed: 4
hal: 4
arab: 4
very: 4
way: 4
2014: 3
442: 2
nice: 2

Recommendations
-----------------------------
- The positive sentiment is low. Review the content style and presentation.
- The negative sentiment is high. Check the most frequent negative words carefully.


In [5]:
%pip install streamlit pyngrok

Note: you may need to restart the kernel to use updated packages.


In [6]:
# (5) Building Streamlit Dashboard

import plotly.express as px
import streamlit as st

st.set_page_config(
    page_title = 'YouTube Sentiment',
    page_icon = '▶️',
    layout = 'wide')

@st.cache_data
def load_data(video_id):
    file_path = f'data/{video_id}_results.csv'
    if not os.path.exists(file_path):
        return None
    return pd.read_csv(file_path)

st.title('▶️ YouTube Comment Sentiment Analyzer')
st.divider()

with st.sidebar:
    st.header('Settings')
    video_id = st.text_input('Video ID:', value = 'Aj-pyJF6ckU')
    load_button = st.button(
        'Load Results',
        type = 'primary',
        use_container_width = True)

df = load_data(video_id)
if df is None:
    st.warning('No saved results found for this video. Run the pipeline first.')
    st.stop()
total = len(df)
positive = (df['sentiment'] == 'positive').sum()
negative = (df['sentiment'] == 'negative').sum()
neutral = (df['sentiment'] == 'neutral').sum()
col1, col2, col3, col4 = st.columns(4)
col1.metric('Total Comments', total)
col2.metric('Positive 😊', f'{positive} ({100 * positive // total}%)')
col3.metric('Negative 😡', f'{negative} ({100 * negative // total}%)')
col4.metric('Neutral 😐', f'{neutral} ({100 * neutral // total}%)')
st.divider()

col_pie, col_bar = st.columns(2)
counts = df['sentiment'].value_counts().reset_index()
counts.columns = ['sentiment', 'count']
color_map = {
    'positive': '#22c55e',
    'negative': '#ef4444',
    'neutral': '#64748b'}

with col_pie:
    pie_chart = px.pie(
        counts,
        values = 'count',
        names = 'sentiment',
        title = 'Sentiment Distribution',
        color_discrete_map = color_map)
    st.plotly_chart(pie_chart, use_container_width = True)

with col_bar:
    bar_chart = px.bar(
        counts,
        x = 'sentiment',
        y = 'count',
        title = 'Number of Comments',
        color = 'sentiment',
        color_discrete_map = color_map,
        text = 'count')

    bar_chart.update_traces(textposition = 'outside')
    bar_chart.update_layout(showlegend = False)

    st.plotly_chart(bar_chart, use_container_width = True)

st.divider()
st.subheader('💬 Comments')
positive_tab, negative_tab, neutral_tab = st.tabs([
    'Positive ✅',
    'Negative ❌',
    'Neutral ➖'])

columns = ['author', 'text', 'score']
with positive_tab:
    st.dataframe(
        df[df['sentiment'] == 'positive'][columns].head(10),
        use_container_width = True)

with negative_tab:
    st.dataframe(
        df[df['sentiment'] == 'negative'][columns].head(10),
        use_container_width = True)

with neutral_tab:
    st.dataframe(
        df[df['sentiment'] == 'neutral'][columns].head(10),
        use_container_width = True)
st.divider()
csv = df.to_csv(index = False, encoding = 'utf-8-sig').encode('utf-8-sig')
st.download_button(
    '⬇️ Download CSV Report',
    data = csv,
    file_name = f'{video_id}_report.csv',
    mime = 'text/csv',
    use_container_width = True)

2026-05-29 17:21:41.729 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 17:21:41.734 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-05-29 17:21:41.737 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 17:21:42.170 
  command:

    streamlit run c:\Users\Esmael\AppData\Local\Programs\Python\Python313\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-05-29 17:21:42.170 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 17:21:42.171 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 17:21:42.172 Thread 'MainThread': missing ScriptRunContext! This war

False

In [11]:
# (6) Running Streamlit Application

import subprocess

os.environ['YOUTUBE_API_KEY'] = os.getenv('YOUTUBE_API_KEY')

process = subprocess.Popen(
    ['streamlit' , 'run' ,'app.py' , '--server.port' , '8501' , '--server.address' , '0.0.0.0' ] ,
    stdout = subprocess.DEVNULL ,
    stderr = subprocess.DEVNULL)
print('Streamlit started')

Streamlit started
